# Session 10: Catch It Out

Click a cell and press **Shift + Enter** to run it. Work from top to bottom.
Cells marked **YOUR TURN** are for you to finish. Put this notebook in the same folder as mood_machine.pth from Session 8. Run the first cell straight away.

In [ ]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2
import matplotlib.pyplot as plt
from emoji_faces import make_face, CLASSES

if torch.accelerator.is_available():
    device = torch.accelerator.current_accelerator()
else:
    device = torch.device("cpu")
print("Using", device)

class EmojiFaces(Dataset):
    def __init__(self, how_many, first_seed, transform=None):
        self.faces = []
        for seed in range(first_seed, first_seed + how_many):
            picture, mood = make_face(seed)
            if transform:
                picture = transform(picture)
            self.faces.append((picture, mood))

    def __len__(self):
        return len(self.faces)

    def __getitem__(self, idx):
        return self.faces[idx]

get_ready = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

print("Drawing faces... (about a minute)")
train_faces = EmojiFaces(12000, first_seed=0, transform=get_ready)
test_faces = EmojiFaces(3000, first_seed=1_000_000, transform=get_ready)
train_loader = DataLoader(train_faces, batch_size=64, shuffle=True)
test_loader = DataLoader(test_faces, batch_size=64)
print(len(train_faces), "training faces and", len(test_faces), "test faces")

class MoodNet(nn.Module):
    def __init__(self, hidden=256):
        super().__init__()
        self.flatten = nn.Flatten()
        self.hidden1 = nn.Linear(3 * 32 * 32, hidden)
        self.hidden2 = nn.Linear(hidden, 128)
        self.output = nn.Linear(128, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.flatten(x)
        x = self.relu(self.hidden1(x))
        x = self.relu(self.hidden2(x))
        return self.output(x)

def train_one_epoch(model, loader, loss_fn, optimiser):
    model.train()
    for faces, moods in loader:
        faces, moods = faces.to(device), moods.to(device)
        guesses = model(faces)            # 1. make guesses
        loss = loss_fn(guesses, moods)    # 2. how wrong were they?
        optimiser.zero_grad()             # 3. wipe old gradients
        loss.backward()                   # 4. find new gradients (back propagation)
        optimiser.step()                  # 5. nudge the weights
    return loss.item()

def check_accuracy(model, loader):
    model.eval()
    correct = 0
    with torch.no_grad():
        for faces, moods in loader:
            faces, moods = faces.to(device), moods.to(device)
            guesses = model(faces).argmax(dim=1)
            correct += (guesses == moods).sum().item()
    return 100 * correct / len(loader.dataset)

import os
model = MoodNet().to(device)
if os.path.exists("mood_machine.pth"):
    model.load_state_dict(torch.load("mood_machine.pth", weights_only=True))
    print("Loaded your saved Mood Machine")
else:
    print("No saved Mood Machine found, so training one now (a few minutes)...")
    torch.manual_seed(1)
    opt = torch.optim.Adam(model.parameters(), lr=0.001)
    for epoch in range(1, 6):
        train_one_epoch(model, train_loader, nn.CrossEntropyLoss(), opt)
        print("  epoch", epoch, "done")
    torch.save(model.state_dict(), "mood_machine.pth")
model.eval()
print("test accuracy:", round(check_accuracy(model, test_loader), 1), "%")

## 1. Read Brand-New Faces

In [ ]:
def read_mood(model, picture):
    x = get_ready(picture).unsqueeze(0).to(device)
    with torch.no_grad():
        chances = torch.softmax(model(x), dim=1)[0]
    best = chances.argmax().item()
    return CLASSES[best], chances[best].item()

plt.figure(figsize=(9, 4))
for i, seed in enumerate(range(5_000_000, 5_000_008)):
    picture, mood = make_face(seed)
    guess, chance = read_mood(model, picture)
    plt.subplot(2, 4, i + 1)
    plt.imshow(picture)
    plt.title(guess + " " + str(round(100 * chance)) + "%", color="green" if guess == CLASSES[mood] else "red")
    plt.axis("off")
plt.show()

## 2. Which Moods Fool It?

In [ ]:
from collections import Counter
right, total, mix_ups = Counter(), Counter(), Counter()
with torch.no_grad():
    for faces, moods in test_loader:
        guesses = model(faces.to(device)).argmax(dim=1).cpu()
        for true_mood, guess in zip(moods.tolist(), guesses.tolist()):
            total[true_mood] += 1
            if guess == true_mood:
                right[true_mood] += 1
            else:
                mix_ups[(CLASSES[true_mood], CLASSES[guess])] += 1
for m in range(10):
    print(CLASSES[m], round(100 * right[m] / total[m]), "%")
print("Most common mix-ups:")
for (real, guess), n in mix_ups.most_common(3):
    print(" ", real, "read as", guess, n, "times")

## 3. Drawn Faces
Download the four test drawings from the website into your mood-machine folder, then run this.

In [ ]:
# PREDICT first: write down what you think the Mood Machine will say for each drawing.
import os
from PIL import Image

def read_drawing(filename):
    picture = Image.open(filename).convert("RGB").resize((32, 32))
    return read_mood(model, picture)

drawings = ["happy_high.png", "happy_low.png", "sad_high.png", "sad_low.png"]
plt.figure(figsize=(10, 3))
for i, name in enumerate(drawings):
    if os.path.exists(name):
        guess, chance = read_drawing(name)
        plt.subplot(1, 4, i + 1)
        plt.imshow(Image.open(name))
        plt.title(name + "\n" + guess + " " + str(round(100 * chance)) + "%", fontsize=10)
        plt.axis("off")
plt.show()

In [ ]:
# YOUR TURN: draw your own faces in Paint and save them in this folder as PNG files.
# Add their names to the list, run the cell, and record what the Mood Machine says.
my_drawings = ["my_face.png"]
for name in my_drawings:
    if os.path.exists(name):
        print(name, read_drawing(name))
    else:
        print(name, "is not in this folder yet")

## 4. Upside Down

In [ ]:
flipped = [(torch.flip(face, dims=[1]), mood) for face, mood in test_faces]
flipped_loader = DataLoader(flipped, batch_size=64)
print("upright faces:", round(check_accuracy(model, test_loader), 1), "%")
print("upside-down faces:", round(check_accuracy(model, flipped_loader), 1), "%")